# Лабораторная работа №4

## Мультимодальный поиск: текст + изображение


### Цель

Построить воспроизводимую мультимодальную поисковую систему на основе общего пространства текстовых и визуальных эмбеддингов, сравнить текстовый и визуальный поиск и исследовать устойчивость retrieval к формулировке запроса.

Результатом работы является поисковый контур с измеренным качеством, скоростью и анализом ограничений мультимодальной модели.

## 1. Что используется в работе

Преподаватель предоставляет:

- датасет изображений с метками и краткими описаниями;
- фиксированное разбиение `gallery / validation queries / test queries`;
- готовую предобученную CLIP-совместимую модель;
- подготовленное окружение с `PyTorch`, `transformers` или `open_clip`, `faiss-cpu`;
- ограничение на вычислительный бюджет.

Модель в обязательной части не дообучается. Основная техническая задача — построить общий индекс, реализовать разные типы запросов, спроектировать протокол оценки и исследовать чувствительность поиска к тексту.

## 2. Краткая теоретическая справка

### 2.1. Общее пространство эмбеддингов

После L2-нормализации релевантность можно оценивать косинусной близостью:

### 2.2. Контрастивное обучение

Модель повышает сходство соответствующих пар «изображение–текст» и понижает сходство несоответствующих. Качество зависит от близости целевого домена и языка запросов к данным предобучения.

### 2.3. Режимы поиска

- **Text-to-image** — текст сравнивается с индексом изображений.
- **Image-to-image** — изображение сравнивается с тем же индексом.
- **Hybrid query** — текстовый и визуальный эмбеддинги комбинируются после нормализации.

### 2.4. Чувствительность к формулировке

На результат влияют язык, длина, порядок признаков, конкретность, лишние детали и доменная терминология. Поэтому запросы оцениваются сериями перефразировок.

## 3. Задачи

1. Проверить датасет и сформировать gallery.
2. Получить и сохранить визуальные эмбеддинги.
3. Построить индекс поиска.
4. Реализовать text-to-image retrieval.
5. Реализовать image-to-image retrieval.
6. Реализовать hybrid query.
7. Сформировать набор текстовых запросов с вариантами формулировок.
8. Оценить Recall@K, Precision@K, mAP, MRR и latency.
9. Провести обязательное исследование чувствительности к формулировке.
10. Провести одно дополнительное исследование.
11. Проанализировать успешные и неуспешные запросы.
12. Сформулировать вывод о пригодности модели для целевого домена.

## 4. Подготовка данных и запросов

Gallery должна содержать уникальные изображения, стабильные идентификаторы, класс или атрибуты и путь к изображению.

Для text-to-image оценки сформируйте не менее **30 смысловых запросов**. Для каждого подготовьте не менее **трёх перефразировок**:

1. краткий запрос;
2. подробный запрос;
3. альтернативная формулировка.

Итого обязательный набор — не менее **90 текстовых запросов**. Каждый запрос должен иметь явно определённый набор релевантных изображений.

In [ ]:
from pathlib import Path
import json
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
try:
    import faiss
except ImportError:
    faiss = None
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("FAISS available:", faiss is not None)

In [ ]:
gallery_table = pd.DataFrame(columns=["image_id", "path", "class_id", "attributes", "split"])
query_table = pd.DataFrame(columns=["query_id", "intent_id", "wording_type", "text", "relevant_image_ids", "split"])
gallery_table.head(), query_table.head()

In [ ]:
def validate_gallery(table: pd.DataFrame) -> None:
    pass

def validate_queries(query_table: pd.DataFrame, gallery_table: pd.DataFrame) -> None:
    pass


**Контрольная точка 1**

До построения индекса должны быть готовы:

- проверенная gallery;
- не менее 30 intent;
- не менее трёх формулировок каждого intent;
- список релевантных изображений для каждого запроса;
- разделение validation/test запросов;
- проверка отсутствия пустых релевантных множеств.

## 5. Модель и эмбеддинги

Используется единая предобученная мультимодальная модель.

```python
image_embeddings = encoder.encode_images(paths)
text_embeddings = encoder.encode_texts(texts)
```

Оба результата должны быть L2-нормализованы и иметь одинаковую размерность.

In [ ]:
try:
    from lab_multimodal import MultimodalEncoder
except ImportError:
    MultimodalEncoder = None
    print("Модуль lab_multimodal должен быть предоставлен преподавателем.")

# encoder = MultimodalEncoder(model_name="...")

In [ ]:
def validate_embeddings(embeddings: np.ndarray) -> None:
    assert embeddings.ndim == 2
    assert np.isfinite(embeddings).all()
    norms = np.linalg.norm(embeddings, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-3)

def extract_gallery_embeddings(encoder, gallery_table: pd.DataFrame, batch_size: int):
    pass


In [ ]:
def save_embedding_pack(path: Path, embeddings: np.ndarray, ids: list[str], metadata: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, embeddings=embeddings.astype(np.float32), ids=np.array(ids), metadata=json.dumps(metadata, ensure_ascii=False))

## 6. Индекс и режимы поиска

Для L2-нормализованных эмбеддингов используйте inner product. Все режимы поиска используют один и тот же индекс изображений.

In [ ]:
def build_index(image_embeddings: np.ndarray):
    if faiss is None:
        raise RuntimeError("Установите faiss-cpu.")
    pass

def search_index(index, query_embeddings: np.ndarray, top_k: int):
    pass


In [ ]:
def text_to_image_search(encoder, index, texts: list[str], top_k: int):
    pass

def image_to_image_search(encoder, index, image_paths: list[str], top_k: int):
    pass

### Hybrid query

Параметр *alpha* должен лежать в диапазоне `[0, 1]`.

In [ ]:
def hybrid_embedding(text_embedding: np.ndarray, image_embedding: np.ndarray, alpha: float) -> np.ndarray:
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must be in [0, 1]")
    combined = alpha * text_embedding + (1.0 - alpha) * image_embedding
    norm = np.linalg.norm(combined, axis=-1, keepdims=True)
    return combined / np.clip(norm, 1e-12, None)

## 7. Метрики и экспериментальный конвейер

Обязательные метрики:

- Recall@1, Recall@5, Recall@10;
- Precision@5;
- mAP;
- MRR;
- mean и p95 latency;
- доля запросов без релевантного результата в top-10;
- устойчивость между перефразировками одного intent.

In [ ]:
def retrieval_metrics(relevant_ids: list[set[str]], retrieved_ids: list[list[str]], ks=(1,5,10)) -> dict:
    pass

def paraphrase_stability(query_table: pd.DataFrame, retrieved_ids_by_query: dict[str, list[str]], top_k: int = 10) -> pd.DataFrame:
    pass


In [ ]:
@dataclass(frozen=True)
class SearchConfig:
    name: str
    mode: str
    top_k: int = 10
    alpha: float | None = None
    language: str = "ru"
    prompt_template: str = "{query}"

required_configs = [
    SearchConfig(name="text_default", mode="text"),
    SearchConfig(name="image_default", mode="image"),
    SearchConfig(name="hybrid_equal", mode="hybrid", alpha=0.5),
]
required_configs

In [ ]:
RUNS_PATH = OUTPUT_DIR / "runs.jsonl"

def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def run_search_experiment(config: SearchConfig, encoder, index, gallery_table: pd.DataFrame, query_table: pd.DataFrame) -> dict:
    raise NotImplementedError

**Контрольная точка 2**

Конвейер считается готовым, если:

- gallery embeddings считаются один раз;
- все режимы используют один индекс;
- результаты привязаны к query_id;
- журнал содержит конфигурацию, метрики и latency;
- повторный запуск пропускает завершённые серии;
- validation и test не смешиваются.

## 8. Обязательное и дополнительное исследование

### Обязательное исследование

Исследуйте чувствительность text-to-image поиска к формулировке запроса: сравните краткие, подробные и альтернативные формулировки; оцените разброс Recall@K и mAP; найдите устойчивые и неустойчивые intent.

### Дополнительное исследование

Выберите один фактор:

- русский против английского;
- шаблоны prompt;
- длина запроса;
- наличие доменной терминологии;
- значение `alpha` в hybrid query;
- иной согласованный фактор.

Дополнительная серия ограничена **2–4 конфигурациями**.

In [ ]:
research_question = ""
hypothesis = ""
extra_configs = []

## 9. Оценка, анализ и сдача

Минимальное сравнение:

1. text-to-image;
2. image-to-image;
3. hybrid query;
4. три типа формулировок;
5. дополнительная серия.

Постройте Recall@K по режимам, mAP по типам формулировок, latency, распределение устойчивости intent и график дополнительного фактора.

In [ ]:
results = pd.DataFrame()
results

In [ ]:
def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    pass


In [ ]:
# TODO:
# 1. Recall@K по режимам;
# 2. mAP по wording_type;
# 3. latency;
# 4. paraphrase stability;
# 5. дополнительный фактор.

### Анализ запросов

Выберите:

- не менее пяти устойчивых intent;
- не менее пяти неустойчивых intent;
- минимум три случая, где image-to-image лучше text-to-image;
- минимум три случая, где hybrid query улучшает результат;
- минимум три случая, где hybrid query ухудшает результат.

Для каждого покажите запрос, top-5 изображений, similarity, релевантность и объяснение результата.

### Обязательные артефакты

1. Проверенная gallery.
2. Query set из не менее 90 запросов.
3. Сохранённые visual embeddings.
4. Поисковый индекс.
5. Text-to-image retrieval.
6. Image-to-image retrieval.
7. Hybrid retrieval.
8. Retrieval-метрики.
9. Метрика устойчивости перефразировок.
10. Журнал запусков.
11. Обязательное исследование prompt sensitivity.
12. Дополнительное исследование.
13. Не менее пяти графиков.
14. Визуализация top-K.
15. Итоговые выводы.

## Критерии оценивания

- Подготовка gallery и query set
- Visual embeddings и индекс
- Text-to-image retrieval
- Image-to-image и hybrid retrieval
- Экспериментальный конвейер
- Исследование prompt sensitivity
- Дополнительное исследование
- Представление результатов
- Анализ и выводы

### Условия зачёта

Работа не засчитывается, если:

- validation и test смешаны;
- релевантные множества не определены;
- приведены только единичные примеры без агрегированных метрик;
- эмбеддинги сравниваются без нормализации и без обоснования;
- выводы не подтверждаются журналом экспериментов.